# 問題
訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [4]:
# 87. ファインチューニング（SST-2, BERT, accuracy評価）
# 必要なライブラリ:
# pip install -U transformers datasets evaluate accelerate

import os
from dataclasses import dataclass
from typing import Dict, List, Union

import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

# 1) 乱数固定（再現性）
set_seed(42)

# 2) データ読み込み（GLUE: SST-2 の train/dev）
#   - label: 0=negative, 1=positive
raw_datasets = load_dataset("sst2")
train_ds = raw_datasets["train"]
dev_ds = raw_datasets["validation"]  # "dev"

# 3) トークナイザ & 前処理
model_name = "bert-base-uncased"  # 任意で他モデルに変更可（roberta-base など）
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

def preprocess(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding=False,   # 動的パディングはデータコラレーターに任せる
        max_length=128,
    )

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=["sentence", "idx"])
tokenized_dev = dev_ds.map(preprocess, batched=True, remove_columns=["sentence", "idx"])

# 4) データコラレーター（動的パディング）
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 5) 評価指標（accuracy）
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# 6) 事前学習済みモデル（分類ヘッド付き）
num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# 7) 学習設定
training_args = TrainingArguments(
    output_dir="./sst2-bert-finetune",
    evaluation_strategy="epoch",     # 各エポック毎に dev で評価
    save_strategy="epoch",
    load_best_model_at_end=True,     # dev 精度ベストを最後にロード
    metric_for_best_model="accuracy",
    greater_is_better=True,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    report_to="none",                # 余計なログ先を無効化
    seed=42,
)

# 8) Trainer 作成 & 学習
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# 9) 開発セットでの最終評価（正解率を表示）
eval_results = trainer.evaluate()
print(f"Validation accuracy: {eval_results['eval_accuracy']:.4f}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'